## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-23-16-49-39 +0000,nyt,"Squeezed for Space in Court Filings, Lawyers G...",https://www.nytimes.com/2026/11/23/us/federal-...
1,2026-11-23-16-47-26 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/23/techno...
2,2026-11-23-16-44-59 +0000,nyt,Stocks Surge After Jensen Huang Is Reinstated ...,https://www.nytimes.com/2026/11/22/business/nv...
3,2026-11-23-16-44-40 +0000,nypost,Ex-NYPD cop charged in GWB road rage murder vi...,https://nypost.com/2026/11/23/us-news/ex-nypd-...
4,2026-11-23-16-42-00 +0000,wsj,"Treasury Yields Rise as Huang Returns as CEO, ...",https://www.wsj.com/finance/investing/u-s-euro...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2717/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
12,huang,28
14,nvidia,27
11,jensen,23
15,ceo,19
207,deal,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
63,2026-11-23-14-17-57 +0000,chitri,Nvidia board reinstates Jensen Huang as CEO ju...,https://www.chicagotribune.com/2026/11/23/nvid...,133
5,2026-11-23-16-40-02 +0000,startribune,Nvidia board reinstates Jensen Huang as CEO ju...,https://www.startribune.com/nvidia-board-unani...,131
160,2026-11-23-09-52-53 +0000,wsj,Stock Market Today: Nvidia Stock Surges as Inv...,https://www.wsj.com/livecoverage/stock-market-...,130
16,2026-11-23-15-53-43 +0000,nypost,Trump suggests Jensen Huang could have moved t...,https://nypost.com/2026/11/23/us-news/trump-su...,125
12,2026-11-23-16-12-26 +0000,nypost,Trump assured Nvidia board Jensen Huang unlike...,https://nypost.com/2026/11/23/us-news/trump-as...,124


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
63,133,2026-11-23-14-17-57 +0000,chitri,Nvidia board reinstates Jensen Huang as CEO ju...,https://www.chicagotribune.com/2026/11/23/nvid...
166,64,2026-11-23-09-17-10 +0000,nypost,Zelensky confirms Ukraine was behind drone att...,https://nypost.com/2026/11/23/world-news/zelen...
283,52,2026-11-22-17-11-00 +0000,cbc,COP31 climate talks collapse after member stat...,https://www.cbc.ca/news/world/cop31-talks-coll...
154,52,2026-11-23-10-00-00 +0000,wsj,Democrats making presidential bids will gather...,https://www.wsj.com/politics/elections/democra...
213,46,2026-11-23-00-00-00 +0000,wsj,Incoming Democratic Sen.-elect Abdul El-Sayed ...,https://www.wsj.com/politics/abdul-el-sayed-is...
244,42,2026-11-22-20-48-02 +0000,chitri,Chuck Hubbard defeats Rep. Virginia Foxx after...,https://www.chicagotribune.com/2026/11/22/chuc...
57,42,2026-11-23-14-28-00 +0000,wsj,The U.S. Army said it plans to detonate severa...,https://www.wsj.com/politics/national-security...
12,40,2026-11-23-16-12-26 +0000,nypost,Trump assured Nvidia board Jensen Huang unlike...,https://nypost.com/2026/11/23/us-news/trump-as...
177,36,2026-11-23-07-23-40 +0000,nypost,Elon Musk reveals new Tesla self-driving techn...,https://nypost.com/2026/11/23/business/elon-mu...
224,35,2026-11-22-22-34-44 +0000,latimes,Missing 19-year-old Marine found dead after we...,https://www.latimes.com/california/story/2026-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
